# Decorators



# Functions are objects

In Python, functions are **first-class objects**. This means that a function can be used in the same way
as any other object (a number, a string, a list):

1. a function can be **assigned** to a variable;
2. a function can be **passed** to another function as an argument;
3. a function can be **returned** from another function;
4. a function can be **stored** in a data structure (a list, a dictionary);
5. a function has **attributes** (`__name__`, `__doc__`, ...).

Decorators are built on these properties.

In [ ]:
def shout(text):
    'Convert the text to upper case'
    return text.upper() + '!'

yell = shout                    # 1. assign to a variable
print(yell('hello'))
print(yell is shout)
print(yell.__name__, '-', yell.__doc__)   # 5. attributes

HELLO!
True
shout - Convert the text to upper case


In [ ]:
def apply(func, value):         # 2. pass as an argument
    return func(value)

print(apply(shout, 'python'))
print(apply(len, 'python'))

actions = {'shout': shout, 'length': len}   # 4. store in a dictionary
print(actions['length']('decorator'))

PYTHON!
6
9


In [ ]:
def make_greeting(greeting):    # 3. return from a function (this is a closure)
    def greet(name):
        return f'{greeting}, {name}!'
    return greet

say_hi = make_greeting('Hi')
print(say_hi('Alice'))

Hi, Alice!


# What is a decorator?

Suppose that for debugging purposes we want to print a logging message every time the sin function is called. We might arrive at something like the following solution.

In [ ]:
import math

def sin(x):
    print(f"The sin function was called with argument {x}")
    return math.sin(x)

print(sin(0))

The sin function was called with argument 0
0.0


Such constructs are called function wrappers: the user-defined sin function wraps the math.sin function.

Now suppose that we want to do the same thing with other functions. Writing our own wrapper for each of them would lead to excessive code duplication. To avoid this, let's write an announce function that, taking as input some function of one argument, func, returns a wrapper function, wrap. The wrapper function wrap, in turn, calls func, but before that it prints its name (func.__name__) and the value of the argument with which it was called.

An implementation of this function might look something like this.

In [ ]:
def announce(func):
    def wrap(x):
        print(f"The function {func.__name__} was called with argument {x}")
        return func(x)
    return wrap

Inside its body, the announce function declares a new function, wrap, and thereby creates an object of that function. Once this object is created, it is immediately returned outward. The wrap function, in turn, is a wrapper: it returns the result of evaluating the wrapped function func, but before that it performs additional actions, namely printing the name of the function (the func.__name__ attribute) and the argument with which the function is called. The func function is called with that same argument.

Let's verify that this function works. To do this, we'll wrap the sine and cosine functions.

In [ ]:
from math import sin, cos

print(sin(0), cos(0))

sin = announce(sin)
cos = announce(cos)

print(sin(0), cos(0))

0.0 1.0
The function sin was called with argument 0
The function cos was called with argument 0
0.0 1.0


It can be seen that wrapping the sin and cos functions produces the desired effect.

Functions of this kind are called decorators.

We can pass not only numbers to an outer function, but also another function!

So closures can be used to change the behaviour of a function quickly.

In Python, a **decorator** is a function (or any other callable object) that takes a function or a class
as its argument and returns a function or a class. It may also take some additional arguments.

In fact, decorators are just special "wrappers". They allow us to run some code before and after the
decorated function without changing the function itself.

Schematically:

```
             decorator
func  ───────────────────►  wrapper
                             ├── code before the call
                             ├── result = func(...)
                             ├── code after the call
                             └── return result
```

In [ ]:
def deprecate(func):
    def inner(*args, **kwargs):
        print('Warning: ' + func.__name__ + ' is deprecated')
        return func(*args, **kwargs)
    return inner

pprint = deprecate(print)

pprint([1, 2, 3], '123', sep='\n')

[1, 2, 3]
123


In [ ]:
mmax = deprecate(max)
mmax(3, 4, 5)

5

Now we get a message before the function is executed. In the same way, we can change the behaviour of our
own functions:

In [ ]:
def own_max(a, b):
    'This is a really nice looking docstring'
    return a if a > b else b

In [ ]:
own_max(3, 4)

4

In [ ]:
new_own_max = deprecate(own_max)
new_own_max(3, 4)

4

In [ ]:
deprecate(own_max)

<function __main__.deprecate.<locals>.inner(*args, **kwargs)>

Note that `deprecate(own_max)` returns a **new** function, `inner`. The original function `own_max` does
not change. `inner` is a closure: it remembers the variable `func` from the enclosing scope.

# The `@` syntax

It is not convenient to wrap a function with an assignment every time, so Python has a more convenient
mechanism: the decorator syntax.

In Python, a decorator is applied to a function or a class with the `@` symbol. These two pieces of code
are **equivalent**:

```python
@decorator
def func():
    ...
```

```python
def func():
    ...

func = decorator(func)
```

Let us look at a simple decorator that prints messages before and after a function call.

In [ ]:
def my_shiny_new_decorator(function_to_decorate):
    # Inside, the decorator defines a "wrapper" function. The wrapper is placed
    # around the decorated function, so it can run any code
    # before and after it.
    def the_wrapper_around_the_original_function():
        print("I am code that runs before the function is called")
        function_to_decorate()  # the function itself
        print("And I am code that runs after it")
    # We return the wrapper
    return the_wrapper_around_the_original_function

# Now imagine that we have a function that we do not want to change anymore.
def stand_alone_function():
    print("I am a simple lonely function. You won't dare to change me, will you?")

In [ ]:
stand_alone_function()

I am a simple lonely function. You won't dare to change me, will you?


In [ ]:
stand_alone_function_decorated = my_shiny_new_decorator(stand_alone_function)
stand_alone_function_decorated()

I am code that runs before the function is called
I am a simple lonely function. You won't dare to change me, will you?
And I am code that runs after it


In [ ]:
stand_alone_function()

I am a simple lonely function. You won't dare to change me, will you?


In [ ]:
@my_shiny_new_decorator
def another_stand_alone_function():
    print("Leave me alone")

In [ ]:
another_stand_alone_function()

I am code that runs before the function is called
Leave me alone
And I am code that runs after it


## When is a decorator applied?

It is important to understand that there are **two different moments**:

1. **Decoration time.** The decorator itself is called **once**, when Python executes the `def` statement
   (usually when the module is imported). At this moment the wrapper is created.
2. **Call time.** The wrapper is called **every time** we call the decorated function.

Let us check this:

In [ ]:
def loud_decorator(func):
    print(f'Decorating {func.__name__}')      # runs once, at decoration time
    def wrapper():
        print(f'Calling {func.__name__}')     # runs at every call
        return func()
    return wrapper

@loud_decorator
def hello():
    print('Hello!')

print('--- the function is defined, but not called yet ---')

Decorating hello
--- the function is defined, but not called yet ---


In [ ]:
hello()
hello()

Calling hello
Hello!
Calling hello
Hello!


# Decorating functions with arguments

Let us pass arguments to the decorated function:

In [ ]:
def a_decorator_passing_arguments(function_to_decorate):
    def a_wrapper_accepting_arguments(arg1, arg2):
        print("Look what I got:", arg1, arg2)
        function_to_decorate(arg1, arg2)
    return a_wrapper_accepting_arguments

# When we call the function returned by the decorator,
# we actually call the "wrapper".
# We pass arguments to the wrapper, and the wrapper passes them
# to the decorated function

@a_decorator_passing_arguments
def print_full_name(first_name, last_name):
    print("My name is", first_name, last_name)

print_full_name("Vasya", "Petrov")

Look what I got: Vasya Petrov
My name is Vasya Petrov


This wrapper works only with functions of **exactly two** positional arguments, and it **loses the
return value** (the wrapper has no `return`). A universal decorator must:

- accept any arguments: `*args, **kwargs`;
- pass them to the original function unchanged;
- **return** the result of the original function.

Here is the general template of a decorator that we will use from now on:

```python
import functools

def decorator(func):
    @functools.wraps(func)            # keeps __name__, __doc__, ... (see below)
    def wrapper(*args, **kwargs):
        # code before the call
        result = func(*args, **kwargs)
        # code after the call
        return result
    return wrapper
```

In [ ]:
def broken(func):
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)          # no return!
    return wrapper

@broken
def add(a, b):
    return a + b

print(add(2, 3))

None


# Stacking decorators

We can apply several decorators to one function. Decorators are **applied from the bottom up** (the one
closest to `def` is applied first), and the wrappers are **executed from the top down**:

```python
@congrats
@deprecated
def own_max(a, b): ...
```

is the same as

```python
own_max = congrats(deprecated(own_max))
```

In [ ]:
def deprecated(func):
    def wrapper(*args, **kwargs):  # Now the inner function is called wrapper, not inner
        print('Warning: ' + func.__name__ + ' is deprecated')
        return func(*args, **kwargs)
    return wrapper

def congrats(func):
    def wrapper(*args, **kwargs):
        print('Congratulations: ' + func.__name__ + ' is an awesome function')
        return func(*args, **kwargs)
    return wrapper


@congrats
@deprecated
def own_max(a, b):
    'This is a really nice looking docstring'
    return a if a > b else b

own_max(1, 2)

Congratulations: wrapper is an awesome function


2

In [ ]:
own_max(3, 2)

Congratulations: wrapper is an awesome function


3

Look at the first line of the output: `congrats` printed the name `wrapper`, not `own_max`. This is
because `congrats` received the wrapper created by `deprecated`, not the original function.

In [ ]:
help(own_max)

Help on function wrapper in module __main__:

wrapper(*args, **kwargs)



In [ ]:
new_own_max = congrats(deprecate(own_max))

In [ ]:
new_own_max(1, 2)

Congratulations: inner is an awesome function
Congratulations: wrapper is an awesome function


2

We get **four** messages. `own_max` was already decorated with `@congrats` and `@deprecated`, so now we
have four wrappers around the original function. Again, the outer wrappers see the names of the inner
wrappers (`inner`, `wrapper`), not `own_max`.

# Keeping the metadata of a function

However, this approach has a problem:

In [ ]:
print(own_max.__name__)
print(own_max.__doc__)

wrapper
None


After decoration, the name `own_max` refers to the wrapper, so we see the name and the docstring
(`None`) of the wrapper. This makes debugging, `help()` and error messages confusing.

**Solution 1** — copy the attributes to the function `wrapper` manually:

In [ ]:
def deprecated(func):
    def wrapper(*args, **kwargs):
        print('Warning: ' + func.__name__ + ' is deprecated')
        return func(*args, **kwargs)
    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    wrapper.__module__ = func.__module__
    # Other necessary attributes can be copied in the same way
    return wrapper

@deprecated
def own_max(a, b):
    'This is a really nice looking docstring'
    return a if a > b else b

own_max(1, 2)

2

In [ ]:
print(own_max.__name__)
print(own_max.__doc__)

own_max
This is a really nice looking docstring


**Solution 2** — use the decorator `wraps` from the module `functools`:

In [ ]:
import functools

def deprecated(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print('Warning: ' + func.__name__ + ' is deprecated')
        return func(*args, **kwargs)
    return wrapper

@deprecated
def own_max(a, b):
    'This is a really nice looking docstring'
    return a if a > b else b

@deprecated
def own_min(a, b):
    'This is a really nice looking docstring also'
    return a if a < b else b

print(own_max(1, 2))
print(own_min(1, 2))

print(own_max.__name__)
print(own_max.__doc__)

2
1
own_max
This is a really nice looking docstring


In [ ]:
help(functools.wraps)

Help on function wraps in module functools:

wraps(wrapped, assigned=('__module__', '__name__', '__qualname__', '__doc__', '__annotations__'), updated=('__dict__',))
    Decorator factory to apply update_wrapper() to a wrapper function
    
    Returns a decorator that invokes update_wrapper() with the decorated
    function as the wrapper argument and the arguments to wraps() as the
    remaining arguments. Default arguments are as for update_wrapper().
    This is a convenience function to simplify applying partial() to
    update_wrapper().



In [ ]:
print(own_min.__name__)
print(own_min.__doc__)

own_min
This is a really nice looking docstring also


`functools.wraps` copies `__module__`, `__name__`, `__qualname__`, `__doc__` and `__dict__` from the
original function to the wrapper. It also adds the attribute `__wrapped__`, which refers to the original
function. **Always use `functools.wraps` in your decorators.**

In [ ]:
print(own_min.__wrapped__)
print(own_min.__wrapped__(1, 2))   # calls the original function, without the warning

<function own_min at 0x109bf1dc0>
1


# Decorators with parameters

Sometimes we want to configure a decorator, for example `@repeat(3)` or `@trace(sys.stderr)`.

The expression after `@` must return a decorator. So `repeat(3)` is a **function that returns a
decorator**. This gives us **three levels** of nested functions:

```python
def repeat(n):                      # level 1: takes the parameters of the decorator
    def decorator(func):            # level 2: takes the function
        @functools.wraps(func)
        def wrapper(*args, **kwargs):   # level 3: takes the arguments of the function
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator
```

The code

```python
@repeat(3)
def hello(): ...
```

is the same as `hello = repeat(3)(hello)`.

Note: `@repeat(3)` and `@repeat` are different. If we write `@repeat` without brackets, the function
`hello` is passed as `n`, and the program does not work as expected.

Let us look at an example:

In [ ]:
import sys

def trace(dest=sys.stderr):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(func.__name__, 'called with args', args, 'kwargs', kwargs, file=dest)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@trace(sys.stdout)
def f(x, test):
    if test > 1:
        return f(x, test / 2)

f('Hi!', test=42)

f called with args ('Hi!',) kwargs {'test': 42}
f called with args ('Hi!', 21.0) kwargs {}
f called with args ('Hi!', 10.5) kwargs {}
f called with args ('Hi!', 5.25) kwargs {}
f called with args ('Hi!', 2.625) kwargs {}
f called with args ('Hi!', 1.3125) kwargs {}
f called with args ('Hi!', 0.65625) kwargs {}


The recursive call `f(x, test / 2)` also goes through the wrapper, because the global name `f` now refers
to the wrapper. That is why every call is printed.

Here is one more example. It shows which arguments are available at each level:

In [ ]:
def decorator_maker_with_arguments(decorator_arg1, decorator_arg2):
    print("I create decorators! And I received these arguments:",
          decorator_arg1, decorator_arg2)
    def my_decorator(func):
        print("I am a decorator. And you still managed to pass these arguments to me:",
              decorator_arg1, decorator_arg2)
        # Do not confuse the arguments of the decorator with the arguments of the function!
        def wrapped(function_arg1, function_arg2):
            print("I am a wrapper around the decorated function.\n"
                  "And I have access to all the arguments\n"
                  "\t- of the decorator: {0} {1}\n"
                  "\t- of the function: {2} {3}\n"
                  "Now I can pass the necessary arguments further."
                  .format(decorator_arg1, decorator_arg2,
                          function_arg1, function_arg2))
            return func(function_arg1, function_arg2)
        return wrapped
    return my_decorator

@decorator_maker_with_arguments("Leonard", "Sheldon")
def decorated_function_with_arguments(function_arg1, function_arg2):
    print("I am a decorated function and I know only about my own arguments: {0}"
          " {1}".format(function_arg1, function_arg2))

I create decorators! And I received these arguments: Leonard Sheldon
I am a decorator. And you still managed to pass these arguments to me: Leonard Sheldon


In [ ]:
decorated_function_with_arguments("Rajesh", "Howard")

I am a wrapper around the decorated function.
And I have access to all the arguments
	- of the decorator: Leonard Sheldon
	- of the function: Rajesh Howard
Now I can pass the necessary arguments further.
I am a decorated function and I know only about my own arguments: Rajesh Howard


# Decorators with state

A wrapper is a closure, so it can **keep state between calls**. There are two common ways to store
state:

1. a variable in the enclosing scope, changed with `nonlocal`;
2. an attribute of the wrapper function, for example `wrapper.calls`.

**Mini-exercise**

Write the decorator **`once(func)`**. <br>
The decorator allows a function to be called only once. The next calls do nothing and return `None`.

In [ ]:
import functools

def once(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal called
        print('called =', called)
        if called < 1:
            called += 1
            return func(*args, **kwargs)

    called = 0
    return wrapper

In [ ]:
@once
def foo(x):
    print('Hi!', x)

In [ ]:
foo(5)

called = 0
Hi! 5


In [ ]:
foo(4)

called = 1


In [ ]:
foo(42)

called = 1


The variable `called` is defined **after** the function `wrapper`, but this is not a problem: the value is
needed only when `wrapper` is called, and by that time the variable already exists.

The second way uses an attribute of the wrapper. The advantage is that the state is visible from outside:

wrapper.calls is wrapper function attribute in Python, which is used to count the number of calls to the decorated function.

To track the number of calls, an attribute `calls` is created in the wrapper, which is initialized to zero and incremented (increased by one) on each call to the wrapper.

Independence of counters. Each decorated function has its own `calls` attribute, so the counters of different functions do not interfere with each other.

In [ ]:
def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@count_calls
def square(x):
    return x * x

square(2)
square(3)
print(square.calls)

2


# Classes as decorators

Decorators do not have to be functions. Any **callable** object can be a decorator. An object of a class
is callable if the class defines the method `__call__`.

In the example below, one object `register` is used as a decorator for several functions and counts
the calls of all of them in one place.

In [ ]:
from collections import Counter

class Register(object):
    def __init__(self):
        self.stat = Counter()

    def __call__(self, func):
        nm = func.__name__
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            self.stat[nm] += 1
            return func(*args, **kwargs)
        return wrapper

    def __str__(self):
        result = 'fname\tcallcount\n'
        for name, count in self.stat.items():
            result += '{}:\t{}\n'.format(name, count)
        return result

register = Register()

In [ ]:
@register
def f(x):
    return x

@register
def q(x):
    return x * 2

In [ ]:
f(1)

1

In [ ]:
q(2)
q(4)
q(2)

4

In [ ]:
f(5)

5

In [ ]:
print(register)

fname	callcount
f:	2
q:	3



In [ ]:
register(f)

<function __main__.f(x)>

Another variant: the class itself is the decorator. Then `@CountCalls` creates an **object** of the class,
and this object replaces the function. Its method `__call__` is executed at every call.

In [ ]:
class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)   # the same as functools.wraps, but for an object
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

@CountCalls
def cube(x):
    return x ** 3

print(cube(2), cube(3))
print(cube.calls, cube.__name__, type(cube))

8 27
2 cube <class '__main__.CountCalls'>


# Decorating classes

A decorator can also be applied to a **class**. In this case, the decorator receives the class and must
return a class (usually the same one, after some changes).

```python
@decorator
class MyClass:
    ...
```

is the same as `MyClass = decorator(MyClass)`.

Class decorators are often used to add methods or attributes, or to register classes.

In [ ]:
def add_repr(cls):
    def __repr__(self):
        fields = ', '.join(f'{k}={v!r}' for k, v in vars(self).items())
        return f'{cls.__name__}({fields})'
    cls.__repr__ = __repr__
    return cls

@add_repr
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

Point(1, 2)

Point(x=1, y=2)

We can also decorate a single **method** of a class. A method is a function, so the usual template
works. The first argument (`self`) simply comes in `*args`:

In [ ]:
def log_method(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f'{type(self).__name__}.{func.__name__} called with {args}')
        return func(self, *args, **kwargs)
    return wrapper

class Calculator:
    @log_method
    def add(self, a, b):
        return a + b

Calculator().add(2, 3)

Calculator.add called with (2, 3)


5

# Decorators from the standard library

You will often meet these ready-made decorators:

| Decorator | Module | What it does |
|---|---|---|
| `@functools.wraps(func)` | `functools` | copies the metadata of `func` to the wrapper |
| `@functools.lru_cache(maxsize=...)` | `functools` | stores the results of calls (memoization) |
| `@functools.cache` | `functools` | the same as `lru_cache(maxsize=None)` (Python 3.9+) |
| `@staticmethod` | built-in | a method that does not receive `self` |
| `@classmethod` | built-in | a method that receives the class `cls` instead of `self` |
| `@property` | built-in | a method that is used like an attribute |
| `@dataclasses.dataclass` | `dataclasses` | generates `__init__`, `__repr__`, `__eq__` for a class |

Let us look at caching. A naive recursive Fibonacci function is very slow, because it calculates the same
values many times. `lru_cache` remembers the results:

In [ ]:
import time

def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

@functools.lru_cache(maxsize=None)
def fib_cached(n):
    return n if n < 2 else fib_cached(n - 1) + fib_cached(n - 2)

start = time.perf_counter()
print(fib(30), f'{time.perf_counter() - start:.3f} s')

start = time.perf_counter()
print(fib_cached(30), f'{time.perf_counter() - start:.6f} s')
print(fib_cached.cache_info())

832040 0.106 s
832040 0.000021 s
CacheInfo(hits=28, misses=31, maxsize=None, currsize=31)


An example of `@property`, `@staticmethod` and `@classmethod` in one class:

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):              # used as an attribute: t.fahrenheit
        return self.celsius * 9 / 5 + 32

    @staticmethod
    def is_valid(celsius):             # does not need self
        return celsius >= -273.15

    @classmethod
    def from_fahrenheit(cls, f):       # an alternative constructor
        return cls((f - 32) * 5 / 9)

t = Temperature.from_fahrenheit(212)
print(t.celsius, t.fahrenheit, Temperature.is_valid(-300))

100.0 212.0 False


The functools module defines a very useful cache decorator that makes it easy to implement memoization. Memoization is aimed at preventing repeated computations when a function is called repeatedly with the same arguments. This is achieved by storing a history (cache) of function calls: during a call, the function records the set of parameters and the return value if that set of parameters is encountered for the first time. If the function encounters a set of parameters that is contained in the call history, then instead of repeating the computations, the function looks at the value in the history and returns it immediately. The functools.cache decorator stores the call history in a dictionary, which means that searching the call history is very fast, but it imposes a restriction on the hashability of the function's parameters.

As an example, let's define a recursive version of a function for computing Fibonacci numbers. For debugging purposes, we'll apply the announce decorator declared above to it.

In [ ]:
from functools import wraps

@announce
def fib(n: int) -> int:
    return fib(n - 1) + fib(n - 2) if n > 1 else 1


fib(5)

The function fib was called with argument 5
The function fib was called with argument 4
The function fib was called with argument 3
The function fib was called with argument 2
The function fib was called with argument 1
The function fib was called with argument 0
The function fib was called with argument 1
The function fib was called with argument 2
The function fib was called with argument 1
The function fib was called with argument 0
The function fib was called with argument 3
The function fib was called with argument 2
The function fib was called with argument 1
The function fib was called with argument 0
The function fib was called with argument 1


8

We see that the call to the fib function with
 also led to 14 more recursive calls to the same function, among which there are many repeats: the call fib(1) alone was made 5 times.

Let's now add the cache decorator to this function as well.

In [ ]:
from functools import cache

@cache
@announce
def fib(n: int) -> int:
    return fib(n - 1) + fib(n - 2) if n > 1 else 1

fib(5)

The function fib was called with argument 5
The function fib was called with argument 4
The function fib was called with argument 3
The function fib was called with argument 2
The function fib was called with argument 1
The function fib was called with argument 0


8

Now the number of calls has been reduced to 6.

Let's see how this affects performance. To do this, we'll redefine this function without any decorators, along with a function that measures time.

In [ ]:
from time import perf_counter

def fib(n: int) -> int:
    return fib(n - 1) + fib(n - 2) if n > 1 else 1


def measure_time(f, n):
    t1 = perf_counter()
    y = f(n)
    t2 = perf_counter()
    print(f"{t2 - t1:7f} секунд. {f.__name__}(n) = {y} при {n=}")

measure_time(fib, 33)
measure_time(fib, 34)
measure_time(fib, 35)

0.799087 секунд. fib(n) = 5702887 при n=33
1.360175 секунд. fib(n) = 9227465 при n=34
2.019374 секунд. fib(n) = 14930352 при n=35


In [ ]:
fib = cache(fib)

measure_time(fib, 35)


0.000002 секунд. wrap(n) = 14930352 при n=35


In [ ]:
import time

def timing(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__}: {round(elapsed, 4)} s")
        return result
    return wrapper

@timing
def calculate_sum(n):
    return sum(range(n))

calculate_sum(1_000_000)



calculate_sum: 0.0244 s


499999500000

# When to use decorators and what to remember

Decorators are useful when the **same additional behaviour** is needed in many functions, and this
behaviour is not part of the main task of these functions. Typical examples:

- **logging** and tracing of calls;
- **measuring** the execution time;
- **caching** results;
- **checking** arguments or access rights;
- **retrying** a function after an error (for example, a network request);
- **registering** functions (for example, URL handlers in web frameworks such as Flask:
  `@app.route('/')`).

Some features of working with decorators:

1. Decorators slightly slow down a function call, because an extra function (the wrapper) is called.
   Do not forget about it.
2. You cannot "undecorate" a function. There are tricks that allow you to create a decorator that can be
   removed from a function (for example, `__wrapped__`), but this is bad practice. It is better to remember:
   once a function is decorated, the decoration is not cancelled.
3. Decorators wrap functions, and this can make debugging more difficult. `functools.wraps` helps here.
4. A decorator is applied to the name **in the module where it is written**. If you import a function
   from another module, you can decorate it only with an explicit assignment:

   ```python
   from module import good_func

   good_func = decorator(good_func)
   ```

# Tasks

In all tasks, use `functools.wraps`, unless the task says otherwise. For tasks with `assert` checks,
a correct solution runs without errors.

###### Task 1. `text_decor`

Write the decorator `text_decor`, which prints the phrases "Hello" and "Goodbye!" around the call of the
decorated function: "Hello" is printed before the call, and "Goodbye!" is printed after it.

In [ ]:
def text_decor(func):
    def inner():
        print("Hello")
        func()
        print("Goodbye!")
    return inner

In [ ]:
@text_decor
def simple_func():
    print('I am a simple function')

simple_func()
# Expected output:
# Hello
# I am a simple function
# Goodbye!

Hello
I am a simple function
Goodbye!


###### Task 2. `repeater`

Write the decorator `repeater`, which calls the decorated function twice.

In [ ]:
def repeater(func):
    def wrap(*args, **kwargs):
      func(*args, **kwargs)
      func(*args, **kwargs)

    return wrap

In [ ]:
@repeater
def multiply(num1, num2):
    print(num1 * num2)

multiply(2, 7)
# Expected output:
# 14
# 14

14
14


###### Task 3. `double_it`

Write the decorator `double_it`, which returns the doubled result of the decorated function.

In [ ]:
import functools
def double_it(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
      return func(*args, **kwargs)*2
    return wrapper

In [ ]:
@double_it
def multiply(num1, num2):
    return num1 * num2

assert multiply(9, 4) == 72
assert multiply(num1=1, num2=5) == 10
assert multiply.__name__ == 'multiply'
print('OK')

OK


###### Task 4. `timer`

The code below is a first attempt to write a decorator that measures the execution time of a function.
It has several problems. Find them and write a correct decorator `timer`:

1. it must work with any arguments;
2. it must return the result of the function;
3. it must keep the name and the docstring of the function;
4. it must print a message like `g took 0.000123 s`.

*Hint:* use `time.perf_counter()` and choose clear names for the functions.

In [ ]:
import time

def f(g):
    def h(arg):
        start = time.time()
        g(arg)
        end = time.time()
        print(end-start)
    return h

@f
def g(n):
    a=5**n
    print(a)
g(1000)

In [ ]:
import functools
import time

def timer(func):
    pass

In [ ]:
@timer
def power(base, n=2):
    'Raise base to the power n'
    return base ** n

assert power(5, n=3) == 125
assert power.__name__ == 'power'
assert power.__doc__ == 'Raise base to the power n'
print('OK')

###### Task 5. Predict the order

Without running the code, write down what it prints. Explain the order of the lines. Then run the cell.

In [ ]:
def first(func):
    print('decorating with first')
    def wrapper():
        print('first: before')
        func()
        print('first: after')
    return wrapper

def second(func):
    print('decorating with second')
    def wrapper():
        print('second: before')
        func()
        print('second: after')
    return wrapper

@first
@second
def hello():
    print('hello')

print('---')
hello()

decorating with second
decorating with first
---
first: before
second: before
hello
second: after
first: after


*Your answer:*

###### Task 6. `add_color`

Write the decorator `is_prime`, that prints "Prime" if the result of the first function is a prime number, and "Composite" otherwise



In [ ]:
def is_prime(func):
    pass

In [ ]:
@is_prime
def sum_three(a, b, c):
    return a + b + c


assert sum_three(1,3,2) == 'Composite'
assert sum_three(3,3,5) == 'Prime'
print('OK')

###### Task 7. `logger`

Write the decorator `logger`, which prints the name of the function, its arguments and its result:

```
add(2, 3) -> 5
greet('Alice', greeting='Hi') -> 'Hi, Alice'
```

*Hint:* use `repr()` for the arguments and the result.

In [ ]:
def logger(func):
    pass

In [ ]:
@logger
def add(a, b):
    return a + b

@logger
def greet(name, greeting='Hello'):
    return f'{greeting}, {name}'

assert add(2, 3) == 5
assert greet('Alice', greeting='Hi') == 'Hi, Alice'
print('OK')

###### Task 8. `repeat(n)` — a decorator with a parameter

Write the decorator `repeat(n)`, which calls the decorated function `n` times and returns a **list** of
all results.

In [ ]:
def repeat(n):
    pass

In [ ]:
import random

@repeat(3)
def roll():
    return random.randint(1, 6)

results = roll()
assert isinstance(results, list) and len(results) == 3
assert all(1 <= r <= 6 for r in results)
assert roll.__name__ == 'roll'
print('OK', results)

###### Task 9. `validate_positive`

Write the decorator `validate_positive`, which checks that **all** positional and keyword arguments of the
function are positive numbers (`int` or `float`, greater than 0). If this is not true, the decorator must
raise a `ValueError` and must **not** call the function.

In [ ]:
def validate_positive(func):
    pass

In [ ]:
@validate_positive
def rectangle_area(width, height):
    return width * height

assert rectangle_area(2, 3.5) == 7.0
for bad_args in [(-1, 2), (2, 0), ('2', 3)]:
    try:
        rectangle_area(*bad_args)
    except ValueError:
        pass
    else:
        raise AssertionError(f'ValueError was expected for {bad_args}')
print('OK')

###### Task 10. `memoize`

Write your own caching decorator `memoize` (without `functools.lru_cache`). It stores the results in a
dictionary: the key is the tuple of positional arguments, the value is the result. The dictionary must
be available as the attribute `cache` of the decorated function.

Compare the speed of `fib(35)` with and without your decorator.

In [ ]:
def memoize(func):
    pass

In [ ]:
@memoize
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

assert fib(35) == 9227465
assert len(fib.cache) == 36
print('OK')

###### Task 11. `retry(times, exceptions)`

Write the decorator with parameters `retry(times=3, exceptions=(Exception,))`. If the function raises one
of the `exceptions`, the decorator calls it again, at most `times` times in total. If all attempts fail,
the last exception is raised again. Other exceptions are not caught.

In [ ]:
def retry(times=3, exceptions=(Exception,)):
    pass

In [ ]:
attempts = 0

@retry(times=4, exceptions=(ConnectionError,))
def unstable():
    global attempts
    attempts += 1
    if attempts < 3:
        raise ConnectionError('network problem')
    return 'success'

assert unstable() == 'success'
assert attempts == 3

@retry(times=2)
def always_fails():
    raise KeyError('no key')

try:
    always_fails()
except KeyError:
    print('OK')

###### Task 12. Function registry

Write a class `Registry` whose object can be used as a decorator. The decorator saves the function in a
dictionary `functions` under the function's name and returns the **original** function (without a
wrapper). Also write the method `call(name, *args, **kwargs)`, which calls a registered function by its
name.

In [ ]:
class Registry:
    pass

In [ ]:
commands = Registry()

@commands
def hello(name):
    return f'Hello, {name}!'

@commands
def total(*numbers):
    return sum(numbers)

assert set(commands.functions) == {'hello', 'total'}
assert commands.call('hello', 'Bob') == 'Hello, Bob!'
assert commands.call('total', 1, 2, 3) == 6
assert hello('Ann') == 'Hello, Ann!'
print('OK')

###### Task 13*. `singleton` — a class decorator

Write the class decorator `singleton`. A class decorated with it can have only **one** object: every call
of the class returns the same object that was created at the first call.

In [ ]:
def singleton(cls):
    pass

In [ ]:
@singleton
class Config:
    def __init__(self):
        self.settings = {}

a = Config()
b = Config()
a.settings['debug'] = True
assert a is b
assert b.settings['debug'] is True
print('OK')

###### Task 14*. A decorator with optional parameters

Write the decorator `debug` that works **both** with and without brackets:

```python
@debug
def f(): ...

@debug(prefix='>>>')
def g(): ...
```

Before each call, it prints `<prefix> calling <function name>`. The default prefix is `'DEBUG:'`.

*Hint:* check whether the first argument of `debug` is callable.

In [ ]:
def debug(func=None, *, prefix='DEBUG:'):
    pass

In [ ]:
@debug
def f():
    return 1

@debug(prefix='>>>')
def g():
    return 2

assert f() == 1 and g() == 2
assert f.__name__ == 'f' and g.__name__ == 'g'
print('OK')